# Multi-Service Stacks

Notebook 15 covered the Compose file syntax. This notebook focuses on real-world patterns for multi-service stacks: compose file overrides for environment-specific config, profiles for optional services, scaling, service dependencies, and working examples you can run.

Topics:
- Compose file merging and overrides (`-f` chaining)
- `compose.override.yaml` — automatic local overrides
- Profiles — optional services activated on demand
- Scaling services and the `--scale` flag
- Init containers with `service_completed_successfully`
- Secrets in Compose
- Full worked example: API + Postgres + Redis + Nginx reverse proxy

## 1. Compose File Merging

You can split your Compose configuration across multiple files and merge them at runtime with `-f`. This is the standard way to manage environment-specific differences without duplicating the whole file.

```bash
# Base + production override
docker compose -f compose.yaml -f compose.prod.yaml up -d

# Base + CI override
docker compose -f compose.yaml -f compose.ci.yaml up -d
```

Merging rules:
- Keys in later files **override** keys in earlier files
- Lists (ports, volumes, environment) are **merged** — items are added, not replaced
- To completely replace a list, use `!reset` (Compose v2.24+)

### Base + override pattern

```yaml
# compose.yaml — base (works in any environment)
services:
  web:
    image: myapp:${TAG:-latest}
    environment:
      LOG_LEVEL: info
    restart: unless-stopped
  db:
    image: postgres:16-alpine
    volumes:
      - pgdata:/var/lib/postgresql/data
volumes:
  pgdata: {}
```

```yaml
# compose.prod.yaml — production overrides only
services:
  web:
    image: myapp:${TAG}           # TAG must be set — no default
    environment:
      LOG_LEVEL: warn             # override to reduce noise
    deploy:
      resources:
        limits:
          memory: 1g
          cpus: '2.0'
  db:
    environment:
      POSTGRES_PASSWORD: ${DB_PASSWORD}  # injected from CI secrets
```

## 2. `compose.override.yaml` — Automatic Local Overrides

Compose automatically merges `compose.override.yaml` (if present) on top of `compose.yaml` without any extra flags. This is designed for developer-specific local configuration that should not be committed:

```yaml
# compose.override.yaml  (in .gitignore)
services:
  web:
    build: .                   # build locally instead of pulling the image
    volumes:
      - .:/app                 # live reload: mount source code
    environment:
      DEBUG: "true"
    ports:
      - "5678:5678"            # debugger port
  db:
    ports:
      - "5432:5432"            # expose db port to host for local inspection
```

```bash
# git-tracked
compose.yaml

# local only, in .gitignore
compose.override.yaml
.env

# committed for reference
.env.example
compose.prod.yaml
compose.ci.yaml
```

## 3. Profiles — Optional Services

Profiles let you define services in the Compose file that are only started when explicitly activated. Services without a profile are always started. Services with a profile are only started when you activate that profile.

```yaml
services:
  web:
    image: myapp          # no profile — always starts
  db:
    image: postgres:16    # no profile — always starts

  # Tools and optional services
  adminer:
    image: adminer        # DB admin UI
    profiles: [tools]
    ports: ["8080:8080"]
    depends_on: [db]

  mailhog:
    image: mailhog/mailhog
    profiles: [tools]
    ports: ["8025:8025"]

  tests:
    image: myapp
    profiles: [ci]
    command: pytest tests/
    depends_on: [db]
```

```bash
# Normal start: only web + db
docker compose up -d

# With database admin tools
docker compose --profile tools up -d

# Run CI tests (activates the 'ci' profile)
docker compose --profile ci run tests
```

## 4. Scaling Services

Compose can run multiple instances of a service:

```bash
# Start 3 instances of the api service
docker compose up -d --scale api=3

# Scale up without restarting other services
docker compose up -d --scale api=5 --no-recreate
```

Constraints:
- **Cannot use `container_name`** when scaling — it must be unique. Remove it from scaled services.
- **Cannot use fixed host port mappings** like `"8080:80"` — port conflicts. Use `"80"` (random host port) or no port mapping (reach via load balancer instead).
- All instances join the same networks and share the same volume mounts.

With multiple instances behind a shared network alias, Docker DNS provides round-robin load distribution:

```yaml
services:
  api:
    image: myapi
    # No fixed host port — accessed via nginx on the same network
    networks:
      backend:
        aliases: [api]   # all instances share this alias
```

## 5. Init Containers

An init container runs a one-shot job before the main service starts — database migrations, schema setup, seed data. Use `condition: service_completed_successfully`:

```yaml
services:
  migrate:
    image: myapp
    command: python manage.py migrate
    depends_on:
      db:
        condition: service_healthy
    restart: on-failure     # retry if migration fails

  web:
    image: myapp
    depends_on:
      migrate:
        condition: service_completed_successfully  # wait for migrations to finish
      db:
        condition: service_healthy

  db:
    image: postgres:16
    healthcheck:
      test: ["CMD", "pg_isready", "-U", "postgres"]
      interval: 5s
      retries: 5
```

Startup order: `db` healthy → `migrate` runs and exits 0 → `web` starts.

## 6. Secrets in Compose

Compose supports file-based secrets — mounted as files into containers at `/run/secrets/<name>`:

```yaml
services:
  web:
    image: myapp
    secrets:
      - db_password
      - api_key
    environment:
      # App reads the secret from the file, not from env
      DB_PASSWORD_FILE: /run/secrets/db_password

secrets:
  db_password:
    file: ./secrets/db_password.txt   # host file content mounted into container
  api_key:
    environment: API_KEY              # read from host env var
```

The secret file is mounted as a read-only tmpfs inside the container — it never lands in the image or in a Docker layer. The application reads the password from the file path rather than from an environment variable.

```python
# Application code reading a secret from file
with open("/run/secrets/db_password") as f:
    password = f.read().strip()
```

## 7. Full Example: Nginx + API + Postgres + Redis

In [ ]:
import os, textwrap

os.makedirs("/tmp/fullstack/nginx", exist_ok=True)

# Nginx config — reverse proxy to the api service
with open("/tmp/fullstack/nginx/default.conf", "w") as f:
    f.write(textwrap.dedent("""\
        server {
            listen 80;
            location / {
                proxy_pass         http://api;
                proxy_set_header   Host $host;
                proxy_set_header   X-Real-IP $remote_addr;
            }
            location /health {
                return 200 'ok';
                add_header Content-Type text/plain;
            }
        }
    """))

# compose.yaml — the full stack
with open("/tmp/fullstack/compose.yaml", "w") as f:
    f.write(textwrap.dedent("""\
        name: fullstack

        services:

          # ── Nginx reverse proxy (public-facing) ───────────────────────────────
          proxy:
            image: nginx:alpine
            ports:
              - "${PROXY_PORT:-8090}:80"
            volumes:
              - ./nginx/default.conf:/etc/nginx/conf.d/default.conf:ro
            networks: [frontend]
            depends_on:
              api:
                condition: service_started
            restart: unless-stopped
            healthcheck:
              test: ["CMD", "wget", "-qO-", "http://localhost/health"]
              interval: 10s
              timeout: 3s
              retries: 3

          # ── API (behind proxy, reaches db and cache) ──────────────────────────
          api:
            image: nginx:alpine         # stand-in for real app
            environment:
              DB_HOST: db
              REDIS_HOST: redis
            networks: [frontend, backend]
            depends_on:
              db:
                condition: service_healthy
              redis:
                condition: service_healthy
            restart: unless-stopped

          # ── Postgres ──────────────────────────────────────────────────────────
          db:
            image: postgres:16-alpine
            environment:
              POSTGRES_USER: app
              POSTGRES_PASSWORD: ${DB_PASSWORD:-dev}
              POSTGRES_DB: appdb
            volumes:
              - pgdata:/var/lib/postgresql/data
            networks: [backend]
            restart: unless-stopped
            healthcheck:
              test: ["CMD", "pg_isready", "-U", "app"]
              interval: 5s
              timeout: 3s
              retries: 5
              start_period: 10s

          # ── Redis cache ───────────────────────────────────────────────────────
          redis:
            image: redis:7-alpine
            volumes:
              - redisdata:/data
            networks: [backend]
            restart: unless-stopped
            healthcheck:
              test: ["CMD", "redis-cli", "ping"]
              interval: 5s
              timeout: 2s
              retries: 3

          # ── Adminer (optional, tools profile) ────────────────────────────────
          adminer:
            image: adminer
            profiles: [tools]
            ports: ["8091:8080"]
            networks: [backend]
            depends_on: [db]

        volumes:
          pgdata: {}
          redisdata: {}

        networks:
          frontend: {}
          backend:
            internal: true
    """))

print("Stack files written")

In [ ]:
# Validate the config
!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    config --quiet && echo "Valid"

!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    config --services

In [ ]:
# Start the stack (adminer is NOT started — no --profile tools)
!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    up -d

import time; time.sleep(12)

!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    ps

In [ ]:
# Traffic flows: external → proxy:8090 → api → (db + redis on internal net)
print("Proxy health check:")
!curl -s http://localhost:8090/health

print("\nThrough proxy to API (nginx stand-in):")
!curl -s -o /dev/null -w "HTTP %{http_code}\n" http://localhost:8090/

# Verify db is NOT reachable from proxy (different network)
print("\nProxy → db (should be blocked — different network):")
import subprocess
r = subprocess.run(
    ["docker", "compose", "-f", "/tmp/fullstack/compose.yaml",
     "--project-directory", "/tmp/fullstack",
     "exec", "proxy", "ping", "-c", "1", "-W", "2", "db"],
    capture_output=True, timeout=5
)
print("BLOCKED" if r.returncode != 0 else "reachable (unexpected)")

In [ ]:
# Check service logs
!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    logs --tail 5 db

In [ ]:
# Tear down — keep volumes (would contain real data in production)
!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    down

# For a clean slate, also remove volumes
!docker compose -f /tmp/fullstack/compose.yaml \
    --project-directory /tmp/fullstack \
    down -v

print("Done")

## 8. Common Multi-Service Patterns

### Sidecar
A helper container attached to a primary service, sharing its network namespace or a volume:

```yaml
services:
  app:
    image: myapp
    volumes: [logs:/var/log/app]
  log-shipper:
    image: fluent-bit
    volumes:
      - logs:/var/log/app:ro    # reads app's log volume
      - ./fluent-bit.conf:/fluent-bit/etc/fluent-bit.conf:ro
volumes:
  logs: {}
```

### Ambassador
A proxy container that stands between your service and an external dependency, handling retries, TLS termination, or protocol translation:

```yaml
services:
  app:
    environment:
      DB_HOST: db-proxy  # talks to the proxy, not the real db directly
  db-proxy:
    image: pgbouncer
    environment:
      DB_HOST: db
  db:
    image: postgres:16
```

### Worker + Queue

```yaml
services:
  web:
    image: myapp
    command: gunicorn app:app
  worker:
    image: myapp              # same image, different command
    command: celery -A tasks worker
    depends_on: [redis]
  redis:
    image: redis:7-alpine
```

## Summary

- **Compose file merging** with `-f base.yaml -f override.yaml` lets you keep environment-specific config separate without duplicating the whole file. Later files override earlier ones.
- **`compose.override.yaml`** is automatically merged on top of `compose.yaml`. Use it for developer-local config (source mounts, debug ports) that should not be committed.
- **Profiles** (`profiles: [tools]`) mark services as optional. Services without a profile always start. Add `--profile tools` to activate optional services.
- **Scaling** with `--scale api=3` starts multiple instances of a service. Remove `container_name` and avoid fixed host port mappings on scaled services.
- **Init containers** use `condition: service_completed_successfully` on `depends_on` — the main service waits until the init container exits with code 0.
- **Compose secrets** mount host files as read-only tmpfs at `/run/secrets/<name>` — more secure than environment variables for passwords.
- **Network isolation pattern:** public services on a frontend network, backend services (db, cache) on an `--internal` backend network. The API bridges both. The proxy only reaches the API.